In [1]:
!hdfs dfs -mkdir -p /user/asih2/ecommerce/raw
!hdfs dfs -mkdir -p /user/asih2/ecommerce/processed

In [2]:
!hdfs dfs -ls -R /user/asih2/ecommerce

drwxr-xr-x   - asih2 supergroup          0 2026-09-09 20:30 /user/asih2/ecommerce/processed
drwxr-xr-x   - asih2 supergroup          0 2026-09-09 20:30 /user/asih2/ecommerce/raw


In [4]:
!ls -lh transaksi_magelang.csv transaksi_yogyakarta.csv transaksi_semarang.csv

-rw-rw-r-- 1 asih2 asih2 13K Sep  3 08:58 transaksi_magelang.csv
-rw-rw-r-- 1 asih2 asih2 12K Sep  3 08:58 transaksi_semarang.csv
-rw-rw-r-- 1 asih2 asih2 13K Sep  3 08:58 transaksi_yogyakarta.csv


In [5]:
!hdfs dfs -put transaksi_magelang.csv /user/asih2/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/asih2/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/asih2/ecommerce/raw/

In [6]:
!hdfs dfs -ls -h /user/asih2/ecommerce/raw

Found 3 items
-rw-r--r--   1 asih2 supergroup     12.0 K 2026-09-09 20:37 /user/asih2/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 asih2 supergroup     11.9 K 2026-09-09 20:37 /user/asih2/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 asih2 supergroup     12.4 K 2026-09-09 20:37 /user/asih2/ecommerce/raw/transaksi_yogyakarta.csv


In [8]:
import pandas as pd
import subprocess
from io import StringIO

def baca_csv_hdfs(path):
    hasil = subprocess.run(
        ["hdfs", "dfs", "-cat", path],
        capture_output=True,
        text=True,
        check=True
    )
    return pd.read_csv(StringIO(hasil.stdout))

df_magelang = baca_csv_hdfs(
    "/user/asih2/ecommerce/raw/transaksi_magelang.csv"
)

df_yogyakarta = baca_csv_hdfs(
    "/user/asih2/ecommerce/raw/transaksi_yogyakarta.csv"
)

df_semarang = baca_csv_hdfs(
    "/user/asih2/ecommerce/raw/transaksi_semarang.csv"
)

In [9]:
df_gabungan = pd.concat(
    [df_magelang, df_yogyakarta, df_semarang],
    ignore_index=True
)

df_gabungan.head()

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


In [10]:
df_gabungan["kota"].value_counts()

kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64

In [11]:
df_gabungan["total_pendapatan"] = (
    df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
)

df_gabungan.head()

,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota,total_pendapatan
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang,50000
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang,175000
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang,175000
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang,150000
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang,700000


In [12]:
ringkasan = (
    df_gabungan
    .groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
)

ringkasan

,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000


In [13]:
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

In [14]:
!hdfs dfs -put data_gabungan_bersih.csv /user/asih2/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/asih2/ecommerce/processed/

In [15]:
!hdfs dfs -ls -h /user/asih2/ecommerce/processed

Found 2 items
-rw-r--r--   1 asih2 supergroup     40.3 K 2026-09-09 20:45 /user/asih2/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 asih2 supergroup        530 2026-09-09 20:45 /user/asih2/ecommerce/processed/ringkasan_kota_kategori.csv
